In [2]:
!nvidia-smi

Sun Sep 14 17:25:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# === Few-shot (training-free) baseline: Nearest Prototype on embeddings ===
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.preprocessing import normalize

# ---- 1) Load participants & embeddings ----
participants_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Autism/participants.csv')
X_participants = participants_df["ParticipantID"].astype(str).values
y_participants = participants_df["Label"].astype(int).values  # 0/1 label per participant

ds = np.load('/content/ASD_vgg16.npz', allow_pickle=True)
X   = ds['X'].astype('float32')   # embeddings
y   = ds['y'].astype('int32')     # 0=TD, 1=ASD
IDs = ds['IDs'].astype(str)       # per-image participant ID

# L2-normalize embeddings for cosine similarity
Z = normalize(X, norm='l2', axis=1)

print("Embeddings:", Z.shape, "| images:", len(y), "| participants:", len(np.unique(IDs)))

# ---- 2) CV setup (participant-wise) ----
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# K-shot settings
K_list = [1, 3, 5, 10]
repeats = 30  # repeat random support sampling for stability
temperature = 0.2  # softmax temperature for converting cosine sims to probs

def few_shot_fold_metrics(Z_train, y_train, Z_test, y_test, K):
    # indices by class in TRAIN set
    tr0 = np.where(y_train == 0)[0]
    tr1 = np.where(y_train == 1)[0]
    if len(tr0) == 0 or len(tr1) == 0:
        return None  # can't form supports

    logits_accum = np.zeros((len(Z_test), 2), dtype='float32')

    for _ in range(repeats):
        # sample K supports per class (with replacement if needed)
        s0 = np.random.choice(tr0, size=K, replace=(K > len(tr0)))
        s1 = np.random.choice(tr1, size=K, replace=(K > len(tr1)))

        P0 = Z_train[s0].mean(axis=0, keepdims=True)  # TD prototype
        P1 = Z_train[s1].mean(axis=0, keepdims=True)  # ASD prototype
        # Re-normalise prototypes (cosine)
        P0 = P0 / (np.linalg.norm(P0, axis=1, keepdims=True) + 1e-8)
        P1 = P1 / (np.linalg.norm(P1, axis=1, keepdims=True) + 1e-8)

        # cosine similarity == dot product after L2 norm
        logit0 = (Z_test @ P0.T) / temperature
        logit1 = (Z_test @ P1.T) / temperature
        logits = np.hstack([logit0, logit1])  # [n_test, 2]
        logits_accum += logits

    logits = logits_accum / float(repeats)
    # softmax to get p(ASD)
    exps = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exps[:, 1] / (exps.sum(axis=1) + 1e-8)

    # metrics
    auroc = roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else np.nan
    auprc = average_precision_score(y_test, probs)
    y_pred = (probs >= 0.5).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    return auroc, auprc, prec, rec

# ---- 3) Run participant-wise 3-fold CV with K-shot evaluation ----
for K in K_list:
    aucs, auprcs, precs, recs = [], [], [], []
    print(f"\n=== {K}-shot evaluation ===")
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_participants, y_participants), 1):
        train_p = set(X_participants[tr_idx])
        test_p  = set(X_participants[te_idx])

        train_mask = np.array([pid in train_p for pid in IDs])
        test_mask  = np.array([pid in test_p for pid in IDs])

        Z_train, y_train = Z[train_mask], y[train_mask]
        Z_test,  y_test  = Z[test_mask],  y[test_mask]

        if Z_test.shape[0] == 0 or Z_train.shape[0] == 0:
            print(f"Fold {fold}: no data — skipped.")
            continue

        res = few_shot_fold_metrics(Z_train, y_train, Z_test, y_test, K)
        if res is None:
            print(f"Fold {fold}: missing a class in train — skipped.")
            continue

        auroc, auprc, prec, rec = res
        aucs.append(auroc); auprcs.append(auprc); precs.append(prec); recs.append(rec)
        auroc_txt = f"{auroc:.3f}" if not np.isnan(auroc) else "n/a"
        print(f"Fold {fold}: AUROC={auroc_txt} | AUPRC={auprc:.3f} | Precision={prec:.3f} | Recall={rec:.3f} "
              f"[train n={len(y_train)}, test n={len(y_test)}]")

    # summary for this K
    def mean_sd(a):
        a = np.array(a, dtype=float)
        a = a[~np.isnan(a)]
        return (np.mean(a), np.std(a)) if len(a) else (np.nan, np.nan)

    m_auc, s_auc = mean_sd(aucs)
    m_prc, s_prc = mean_sd(auprcs)
    m_p, s_p = mean_sd(precs)
    m_r, s_r = mean_sd(recs)
    auc_txt = f"{m_auc:.3f} ± {s_auc:.3f}" if not np.isnan(m_auc) else "n/a"
    print(f"== {K}-shot Summary ==  AUROC: {auc_txt} | AUPRC: {m_prc:.3f} ± {s_prc:.3f} | "
          f"Precision: {m_p:.3f} ± {s_p:.3f} | Recall: {m_r:.3f} ± {s_r:.3f}")


Embeddings: (3282, 512) | images: 3282 | participants: 54

=== 1-shot evaluation ===
Fold 1: AUROC=0.873 | AUPRC=0.793 | Precision=0.405 | Recall=0.976 [train n=1836, test n=942]
Fold 2: AUROC=0.656 | AUPRC=0.598 | Precision=0.390 | Recall=0.764 [train n=1962, test n=816]
Fold 3: AUROC=0.814 | AUPRC=0.693 | Precision=0.334 | Recall=0.882 [train n=1758, test n=1020]
== 1-shot Summary ==  AUROC: 0.781 ± 0.092 | AUPRC: 0.695 ± 0.079 | Precision: 0.377 ± 0.031 | Recall: 0.874 ± 0.086

=== 3-shot evaluation ===
Fold 1: AUROC=0.868 | AUPRC=0.786 | Precision=0.453 | Recall=0.955 [train n=1836, test n=942]
Fold 2: AUROC=0.669 | AUPRC=0.607 | Precision=0.423 | Recall=0.659 [train n=1962, test n=816]
Fold 3: AUROC=0.793 | AUPRC=0.667 | Precision=0.495 | Recall=0.602 [train n=1758, test n=1020]
== 3-shot Summary ==  AUROC: 0.777 ± 0.082 | AUPRC: 0.687 ± 0.075 | Precision: 0.457 ± 0.029 | Recall: 0.739 ± 0.155

=== 5-shot evaluation ===
Fold 1: AUROC=0.880 | AUPRC=0.801 | Precision=0.485 | Recall=

In [10]:
# === Supervised Contrastive projection + K-shot evaluation (participant-wise CV) ===
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.utils import to_categorical
K = tf.keras.backend

# ------------------ Load participants & embeddings ------------------
# Adjust these two paths if needed:
PARTICIPANTS_CSV = '/content/drive/MyDrive/Colab Notebooks/Autism/participants.csv'
NPZ_PATH         = '/content/ASD_vgg16.npz'

participants_df = pd.read_csv(PARTICIPANTS_CSV)
X_participants = participants_df["ParticipantID"].astype(str).values
y_participants = participants_df["Label"].astype(int).values  # 0/1 per participant

ds = np.load(NPZ_PATH, allow_pickle=True)
X   = ds['X'].astype('float32')   # embeddings
y   = ds['y'].astype('int32')     # 0=TD, 1=ASD
IDs = ds['IDs'].astype(str)       # per-image participant ID

print("Embeddings:", X.shape, "| images:", len(y), "| participants:", len(np.unique(IDs)))

# Optional: L2-normalize base embeddings (often helps cosine-based methods)
X_base = normalize(X, norm='l2', axis=1).astype('float32')

# ------------------ SupCon custom layer ------------------
class SupConLossLayer(layers.Layer):
    def __init__(self, temperature=0.2, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature

    def call(self, inputs):
        z, y_onehot = inputs  # z: [B,d] L2-normalized; y_onehot: [B,2]
        eps = 1e-8
        sim = tf.matmul(z, z, transpose_b=True) / self.temperature  # cosine sim since z is normalized

        bsz = tf.shape(z)[0]
        eye = tf.eye(bsz)
        not_self = 1.0 - eye
        pos_mask = tf.matmul(y_onehot, y_onehot, transpose_b=True) * not_self  # same-class, not self

        exp_sim = tf.exp(sim) * not_self
        pos_sum = tf.reduce_sum(exp_sim * pos_mask, axis=1, keepdims=True) + eps
        denom   = tf.reduce_sum(exp_sim, axis=1, keepdims=True) + eps
        pos_cnt = tf.maximum(tf.reduce_sum(pos_mask, axis=1, keepdims=True), eps)

        loss_i = - tf.math.log(pos_sum / denom) / pos_cnt
        self.add_loss(tf.reduce_mean(loss_i))
        return z  # pass through embeddings

def build_supcon_head(feat_dim=512, emb_dim=128, temperature=0.2, dropout=0.2):
    feat_in = Input(shape=(feat_dim,), name='feat_in')
    y_in    = Input(shape=(2,), name='y_in')  # one-hot labels (for loss only)

    z = layers.Dense(256, activation='relu')(feat_in)
    if dropout and dropout > 0:
        z = layers.Dropout(dropout)(z)
    z = layers.Dense(emb_dim, use_bias=False)(z)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1), name='z_norm')(z)

    z_out = SupConLossLayer(temperature=temperature, name='supcon_loss')([z, y_in])
    model = Model([feat_in, y_in], z_out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=None)

    # Inference model: project features -> normalized embedding
    embed = Model(feat_in, z)
    return model, embed

# ------------------ Balanced tf.data dataset (TRAIN only) ------------------
def make_balanced_dataset(X_train, y_train, batch_size=64):
    feat_dim = X_train.shape[1]
    cls0 = np.where(y_train == 0)[0]
    cls1 = np.where(y_train == 1)[0]
    half = batch_size // 2
    rep0 = (half > len(cls0))
    rep1 = (half > len(cls1))

    def gen():
        while True:
            i0 = np.random.choice(cls0, size=half, replace=rep0)
            i1 = np.random.choice(cls1, size=half, replace=rep1)
            idx = np.concatenate([i0, i1])
            np.random.shuffle(idx)
            xb = X_train[idx].astype('float32')
            yb = to_categorical(y_train[idx], num_classes=2).astype('float32')
            yield xb, yb  # two tensors matching the model's two inputs

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(None, feat_dim), dtype=tf.float32),  # xb
            tf.TensorSpec(shape=(None, 2),        dtype=tf.float32),  # yb (one-hot)
        ),
    )
    # Keras expects a tuple whose first element is the "inputs" tuple
    ds = ds.map(lambda xb, yb: ((xb, yb),))
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# ------------------ K-shot evaluation (cosine prototypes) ------------------
def kshot_eval(Z_train, y_train, Z_test, y_test, K_support=5, repeats=30, temperature=0.2):
    tr0 = np.where(y_train == 0)[0]
    tr1 = np.where(y_train == 1)[0]
    if len(tr0) == 0 or len(tr1) == 0 or len(Z_test) == 0:
        return None

    logits_accum = np.zeros((len(Z_test), 2), dtype='float32')
    for _ in range(repeats):
        s0 = np.random.choice(tr0, size=K_support, replace=(K_support > len(tr0)))
        s1 = np.random.choice(tr1, size=K_support, replace=(K_support > len(tr1)))

        P0 = Z_train[s0].mean(axis=0, keepdims=True); P0 /= (np.linalg.norm(P0, axis=1, keepdims=True) + 1e-8)
        P1 = Z_train[s1].mean(axis=0, keepdims=True); P1 /= (np.linalg.norm(P1, axis=1, keepdims=True) + 1e-8)

        logit0 = (Z_test @ P0.T) / temperature
        logit1 = (Z_test @ P1.T) / temperature
        logits_accum += np.hstack([logit0, logit1])

    logits = logits_accum / float(repeats)
    exps = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exps[:, 1] / (exps.sum(axis=1) + 1e-8)

    auroc = roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else np.nan
    auprc = average_precision_score(y_test, probs)
    y_pred = (probs >= 0.5).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    return auroc, auprc, prec, rec

# ------------------ CV + train SupCon per fold + K-shot eval ------------------
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
K_list = [1, 3, 5, 10]
repeats = 30
temperature = 0.2

# Training params (you can bump for more gains)
batch_size = 64
steps_per_epoch = 200   # try 400–600 for stronger head
epochs = 20             # try 30–40 if you have time
emb_dim = 128

for Kshot in K_list:
    print(f"\n=== SupCon + {Kshot}-shot evaluation ===")
    aucs, auprcs, precs, recs = [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_participants, y_participants), 1):
        train_p = set(X_participants[tr_idx]); test_p = set(X_participants[te_idx])
        train_mask = np.array([pid in train_p for pid in IDs])
        test_mask  = np.array([pid in test_p for pid in IDs])

        X_tr, y_tr = X_base[train_mask], y[train_mask]
        X_te, y_te = X_base[test_mask],  y[test_mask]

        if X_tr.shape[0] == 0 or X_te.shape[0] == 0:
            print(f"Fold {fold}: no data — skipped.")
            continue
        if len(np.unique(y_tr)) < 2:
            print(f"Fold {fold}: train has single class — skipped.")
            continue

        # Build & train SupCon head on TRAIN images only
        K.clear_session()
        supcon_model, embed_model = build_supcon_head(
            feat_dim=X_tr.shape[1], emb_dim=emb_dim, temperature=temperature, dropout=0.2
        )
        ds = make_balanced_dataset(X_tr, y_tr, batch_size=batch_size)
        supcon_model.fit(ds, steps_per_epoch=steps_per_epoch, epochs=epochs, verbose=1)

        # Project to learned space (already L2-normalized by z_norm)
        Z_tr = embed_model.predict(X_tr, batch_size=512, verbose=0)
        Z_te = embed_model.predict(X_te, batch_size=512, verbose=0)

        res = kshot_eval(Z_tr, y_tr, Z_te, y_te, K_support=Kshot, repeats=repeats, temperature=temperature)
        if res is None:
            print(f"Fold {fold}: eval skipped (class missing or empty).")
            continue

        auroc, auprc, prec, rec = res
        aucs.append(auroc); auprcs.append(auprc); precs.append(prec); recs.append(rec)
        auroc_txt = f"{auroc:.3f}" if not np.isnan(auroc) else "n/a"
        print(f"Fold {fold}: AUROC={auroc_txt} | AUPRC={auprc:.3f} | Precision={prec:.3f} | Recall={rec:.3f} "
              f"[train n={len(y_tr)}, test n={len(y_te)}]")

    # Summary for this K
    def mean_sd(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        return (np.mean(arr), np.std(arr)) if len(arr) else (np.nan, np.nan)

    m_auc, s_auc = mean_sd(aucs)
    m_prc, s_prc = mean_sd(auprcs)
    m_p, s_p     = mean_sd(precs)
    m_r, s_r     = mean_sd(recs)
    auc_txt = f"{m_auc:.3f} ± {s_auc:.3f}" if not np.isnan(m_auc) else "n/a"
    print(f"== SupCon + {Kshot}-shot Summary ==  AUROC: {auc_txt} | AUPRC: {m_prc:.3f} ± {s_prc:.3f} | "
          f"Precision: {m_p:.3f} ± {s_p:.3f} | Recall: {m_r:.3f} ± {s_r:.3f}")


Embeddings: (3282, 512) | images: 3282 | participants: 54

=== SupCon + 1-shot evaluation ===
Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0136
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0016
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.1867e-04
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.4332e-04
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.5100e-04
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4289e-04
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.3389e-04
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.1576e-04
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1674e-04
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.8087e-04
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7419e-04
Epoch 12/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0326e-04
Epoch 13/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss

Fold 3: AUROC=0.765 | AUPRC=0.612 | Precision=0.717 | Recall=0.423 [train n=1758, test n=1020]
== SupCon + 1-shot Summary ==  AUROC: 0.764 ± 0.063 | AUPRC: 0.660 ± 0.068 | Precision: 0.662 ± 0.063 | Recall: 0.507 ± 0.164

=== SupCon + 3-shot evaluation ===
Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0140
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0017
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.3480e-04
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.5572e-04
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.5596e-04
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.8505e-04
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.2152e-04
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.7733e-04
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.5422e-04
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.1809e-04
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━